In [1]:
import pandas as pd
import numpy
import seaborn

# Data breakdown
pitching.csv
------------
- gid: Game ID
- id: Player ID
- team: Team ID
- p_seq: order in which the pitcher appeared (1 = starter, 2 = 2nd pitcher of game, etc.)
- stattype: 'value', 'official', 'lower', or 'upper': 'upper' and 'lower' bounds are shown for some stats to highlight data uncertainty 'official' identifies discrepancies between Retrosheet accounts and official statistics
- p_ipouts: outs recorded (IP * 3)
- p_noout: batters faced by in any innings in which no outs were recorded
- p_bfp: batters faced by pitcher
- p_h: hits
- p_d: doubles
- p_t: triples
- p_hr: home runs
- p_r: runs
- p_er: earned runs
- p_w: walks allowed
- p_iw: intentional walks
- p_k: strikeouts
- p_hbp: hit by pitches
- p_wp: wild pitches
- p_bk: balks allowed
- p_sh: sacrifice hits
- p_sf: sacrifice flies
- p_sb: stolen bases allowed
- p_cs: caught stealing allowed
- p_pb: passed balls allowed
- wp: winning pitcher?
- lp: losing pitcher?
- save: did player earn a save?
- gs: game started
- gf: game finished (in relief)
- cg: complete game
- date: date of game
- number: game number (0 if single game; 1, 2, etc. if multiple games on same date)
- site: location of game
- vishome: 'v' if team was visiting team; 'h' if team was home team
- opp: team's opponent in game
- win: equal to one if team won the game
- loss: equal to one if team lost the game
- tie: equal to one if game ended in a tie
- gametype: type of game (e.g., regular-season, exhibition, etc.)
- box: do we have a box score; blank indicates 'no'
- pbp: do we have a play-by-play account; d = deduced account, y = play-by-play account from newspaper or scorecard; blank indicates 'no'

batting.csv
-----------
- gid: Game ID
- id: Player ID
- team: Team ID
- b_lp: player's lineup position (if known)
- b_seq: order in which the player appears (1 = starter, 2 = 2nd player at given lineup slot, etc.)
- stattype: 'value', 'official', 'lower', or 'upper': 'upper' and 'lower' bounds are shown for some stats to highlight data uncertainty 'official' identifies discrepancies between Retrosheet accounts and official statistics
- b_pa: plate appearances
- b_ab: at bats
- b_r: runs scored
- b_h: hits
- b_d: doubles
- b_t: triples
- b_hr: home runs
- b_rbi: runs batted in
- b_sh: sacrifice hits
- b_sf: sacrifice flies
- b_hbp: times hit by pitch
- b_w: walks
- b_iw: intentional walks
- b_k: strikeouts
- b_sb: stolen bases
- b_cs: times caught stealing
- b_gdp: times grounded in double play by team's batters
- b_xi: times reached base on catcher's interference
- b_roe: times reached base via error
- dh: did the player DH in the game
- ph: did the player pinch hit
- pr: did the player pinch run
- date: date of game
- number: game number (0 if single game; 1, 2, etc. if multiple games on same date)
- site: location of game
- vishome: 'v' if team was visiting team; 'h' if team was home team
- opp: team's opponent in game
- win: equal to one if team won the game
- loss: equal to one if team lost the game
- tie: equal to one if game ended in a tie
- gametype: type of game (e.g., regular-season, exhibition, etc.)
- box: do we have a box score; blank indicates 'no'
- pbp: do we have a play-by-play account; d = deduced account, y = play-by-play account from newspaper or scorecard; blank indicates 'no'

In [2]:
def parse_data(path, date_col="date", date_format="%Y%m%d", start_date="2010-04-01"):
    df = pd.read_csv(path)
    df[date_col] = pd.to_datetime(df[date_col], format=date_format)
    df = df.loc[df[date_col] >= start_date]
    return df


pitching_df = parse_data("../data/pitching.csv")
batting_df = parse_data("../data/batting.csv")

# teamstats_df = parse_data("../data/teamstats.csv")
# players_df = parse_data("../data/allplayers.csv", "season", "%Y", "2010-01-01")

gameinfo_df = parse_data("../data/gameinfo.csv")

/tmp/ipykernel_8559/3645819752.py:2: DtypeWarning: Columns (38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/tmp/ipykernel_8559/3645819752.py:2: DtypeWarning: Columns (10,11,13,17,19,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


# Team Composition
The following is a presentation of team composition over the past decade (2010 to present). We highlight the composition of Major League Baseball (MLB) teams by offensive (homeruns, batting averages, etc) and defensive (saves, strikeouts, etc.) properties, and presenting the composition distribution across each season.

In [17]:
class HitStats:
    _metrics_ = {
        'At Bats' : 'b_ab',
        'Hits' : 'b_h',
        'Homeruns' : 'b_hr',
        'RBI' : 'b_rbi',
        'Runs' : 'b_r',
        'Stolen Bases' : 'b_sb',
    }
    def __init__(self, bat_df, game_df):
        self.bat_df = bat_df
        self.game_df = game_df
        self.full_df = self.bat_df.merge(
            self.game_df, 
            on="gid", 
            suffixes=("_p", "_g")
        )

    def calc_metric(self, metric, metric_func=sum):
        m = _metrics_.get(metric)
        if m:
            return self.full_df.groupby([
                    "season",
                    "id"
                ]).apply(
                    lambda x: x[m].metric_func()
                )
        else:
            print(f"Invalid metric, use the following: {_metrics_.keys()}")

In [20]:
class PitchStats:
    def __init__(self, pitch_df, game_df):
        self.pitch_df = pitch_df
        self.game_df = game_df
        self.full_df = self.pitch_df.merge(
            self.game_df, 
            on="gid", 
            suffixes=("_p", "_g")
        )

    # Wins
    def calc_pitch_wins(self):
        return self.full_df.groupby([
                "season",
                "id"
            ]).apply(
                lambda x: x["wp"].sum()
            )
    
    # Losses
    def calc_pitch_losses(self):
        return self.full_df.groupby([
                "season",
                "id"
            ]).apply(
                lambda x: x["lp"].sum()
            )
    
    # ERA
    def calc_era(self):
        self.full_df["era"] = self.full_df["p_er"] / 9.0
        return self.full_df.groupby([
                "season",
                "id"
            ]).apply(
                lambda x: x["era"].mean()
            )
    
    # Strikeout Percentage
    def calc_strikeout_rate(self):
        self.full_df["strikeout_rate"] = self.full_df["p_k"] / self.full_df["p_bfp"]
        return self.full_df.groupby([
                "season",
                "id"
            ]).apply(
                lambda x: x["strikeout_rate"].mean()
            )
    
    # HR/9
    def calc_hr_rate(self):
        self.full_df["hr_rate"] = self.full_df["p_hr"] / 9.0
        return self.full_df.groupby([
                "season",
                "id"
            ]).apply(
                lambda x: x["hr_rate"].mean()
            )
    
    # BB%
    def calc_walk_rate(self, pitch_df, game_df):
        self.full_df["walk_rate"] = self.full_df["p_iw"] + self.full_df["p_w"]
        return self.full_df.groupby([
                "season",
                "id"
            ]).apply(
                lambda x: x["walk_rate"] / x.count()
            )


In [21]:
p = PitchStats(pitching_df, gameinfo_df)
b = HitStats(batting_df, gameinfo_df)

In [22]:
p.calc_strikeout_rate()

season  id      
2010    aardd001    0.254133
        abadf001    0.166450
        accaj001    0.095238
        aceva001    0.030000
        acosm001    0.317509
                      ...   
2025    zefer001    0.281042
        zerpa001    0.235153
        zimmb002    0.035714
        zubet001    0.280952
        zuluy001    0.129252
Length: 12302, dtype: float64

# Team Composition
## Pitching lineup
Includes breakdown of per-pitcher performance (with faceting by type i.e. reliever vs closer vs starting) and historical comparison to present.